# Module 5A - Final Assignments

Two hands-on assignments that test everything you learned in Module 5A:

| Assignment | Focus | Difficulty | Concepts Tested |
|---|---|---|---|
| **Assignment 1** | SQL & Dashboard | Beginner-Intermediate | SQL Editor, joins, window functions, aggregations, visualizations, alerts |
| **Assignment 2** | RAG Agent | Advanced | Vector Search, tool calling, retrieval gating, PII guardrails, multi-step routing, LLM-as-judge evaluation, Unity Catalog governance |

> **Instructions**: Each assignment has a scenario, specific tasks with constraints, and a validation cell. Complete all tasks, then run the validation cell to check your work. Placeholder cells marked `# YOUR CODE HERE` are where you write your solution.
>
> **Free Edition note**: You have one pre-provisioned Serverless Starter Warehouse for SQL tasks. Use serverless compute for Python tasks.

---
# Assignment 1: SQL Analytics Dashboard

## Scenario
You are a data analyst at **TechMart**, an e-commerce company. The business team needs a sales analytics dashboard to track product performance, customer behavior, and revenue trends.

Your manager has given you raw data in three tables. You must build a SQL analytics layer and a dashboard-ready summary table.

## Constraints
Your solution MUST meet ALL of the following:
1. **Use at least one JOIN** across two or more tables
2. **Use at least one window function** (e.g., `RANK()`, `LAG()`, `SUM() OVER()`, `ROW_NUMBER()`)
3. **Filter by a date range** (e.g., last 30 days, last quarter)
4. **Calculate at least one derived metric** (e.g., profit margin, revenue growth %, customer lifetime value)
5. **Store the final result in a Unity Catalog table** (not a temp view)
6. **Write an alert query** that would trigger if daily revenue drops below a threshold

## Deliverables
* Task 1: Exploratory query with JOIN + window function
* Task 2: Summary table with derived metrics, stored in UC
* Task 3: Visualization from the summary table
* Task 4: Alert query for revenue monitoring
* Run the validation cell to check constraints are met

In [0]:
%sql
-- SETUP: Run this cell to create the sample data for Assignment 1
-- Creates: TechMart orders, customers, and products tables in Unity Catalog

CREATE CATALOG IF NOT EXISTS assignment_techmart;
CREATE SCHEMA IF NOT EXISTS assignment_techmart.analytics;

-- Customers table
CREATE OR REPLACE TABLE assignment_techmart.analytics.customers (
  customer_id   INT,
  customer_name STRING,
  region        STRING,
  signup_date   DATE,
  tier          STRING  -- Bronze, Silver, Gold
);

INSERT INTO assignment_techmart.analytics.customers VALUES
(1, 'Alice Corp',     'North America', '2024-01-15', 'Gold'),
(2, 'Bob Industries', 'Europe',        '2024-03-22', 'Silver'),
(3, 'Charlie LLC',     'Asia Pacific',  '2024-06-10', 'Bronze'),
(4, 'Delta Inc',       'North America', '2024-09-05', 'Gold'),
(5, 'Echo Partners',   'Europe',        '2025-01-20', 'Silver'),
(6, 'Foxtrot Ltd',     'Asia Pacific',  '2025-03-18', 'Bronze'),
(7, 'Golf Associates', 'North America', '2025-05-12', 'Gold'),
(8, 'Hotel Co',        'Europe',        '2025-07-30', 'Silver');

-- Products table
CREATE OR REPLACE TABLE assignment_techmart.analytics.products (
  product_id    INT,
  product_name  STRING,
  category      STRING,
  unit_price    DECIMAL(10,2),
  unit_cost     DECIMAL(10,2)
);

INSERT INTO assignment_techmart.analytics.products VALUES
(101, 'Pro License',      'Software',   499.00, 200.00),
(102, 'Enterprise License', 'Software', 1499.00, 600.00),
(103, 'Vector Search',     'AI Service', 299.00, 100.00),
(104, 'AI Gateway',       'AI Service', 199.00,  80.00),
(105, 'Support Plan',      'Service',    99.00,  40.00),
(106, 'Training Credits',  'Service',    49.00,  20.00);

-- Orders table (90 days of data)
CREATE OR REPLACE TABLE assignment_techmart.analytics.orders AS
SELECT
  CAST(o.order_id AS INT) AS order_id,
  o.customer_id,
  o.product_id,
  o.quantity,
  o.order_date,
  o.order_date AS ship_date,
  CAST(o.quantity * p.unit_price AS DECIMAL(12,2)) AS revenue,
  CAST(o.quantity * p.unit_cost AS DECIMAL(12,2)) AS cost
FROM (
  SELECT
    explode(sequence(1, 200)) AS order_id,
  (c.customer_id),
  (p.product_id),
  CAST(rand() * 10 + 1 AS INT) AS quantity,
  date_add('2025-07-01', CAST(rand() * 90 AS INT)) AS order_date
  FROM assignment_techmart.analytics.customers c
  CROSS JOIN assignment_techmart.analytics.products p
  ORDER BY rand()
  LIMIT 200
) o
JOIN assignment_techmart.analytics.products p ON o.product_id = p.product_id;

SELECT 'Setup complete!' AS status, count(*) AS total_orders FROM assignment_techmart.analytics.orders;

In [0]:
%sql
-- TASK 1: Exploratory Query with JOIN + Window Function
-- PASTE YOUR CODE BELOW


In [0]:
%sql
-- TASK 2: Summary Table with Derived Metrics
-- PASTE YOUR CODE BELOW

##### TASK 3: Visualization
###### Paste the screenshots of Dashboard below

In [0]:
%sql
-- TASK 4: Alert Query for Revenue Monitoring
-- PASTE YOUR CODE BELOW

In [0]:
# VALIDATION: Run this cell AFTER completing all 4 tasks to check constraints.
# This cell verifies that your solution meets all the requirements.

print("=" * 60)
print("ASSIGNMENT 1 - VALIDATION CHECKS")
print("=" * 60)

checks_passed = 0
checks_total = 0

# Check 1: Customer summary table exists in UC
checks_total += 1
try:
    table_exists = spark.catalog.tableExists("assignment_techmart.analytics.customer_summary")
    if table_exists:
        print("[PASS] Task 2: UC table 'assignment_techmart.analytics.customer_summary' exists")
        checks_passed += 1
    else:
        print("[FAIL] Task 2: UC table 'assignment_techmart.analytics.customer_summary' does not exist")
except Exception as e:
    print(f"[FAIL] Task 2: Error checking table - {e}")

# Check 2: Summary table has derived metrics (profit_margin or similar)
checks_total += 1
try:
    if table_exists:
        cols = [f.name for f in spark.table("assignment_techmart.analytics.customer_summary").schema.fields]
        has_margin = any('margin' in c.lower() or 'growth' in c.lower() or 'aov' in c.lower() or 'avg_order' in c.lower() for c in cols)
        if has_margin:
            print(f"[PASS] Task 2: Derived metric found in columns: {cols}")
            checks_passed += 1
        else:
            print(f"[FAIL] Task 2: No derived metric (margin/growth/aov) found in columns: {cols}")
    else:
        print("[SKIP] Task 2: Table doesn't exist, cannot check columns")
except Exception as e:
    print(f"[FAIL] Task 2: Error checking columns - {e}")

# Check 3: Summary table has data
checks_total += 1
try:
    if table_exists:
        row_count = spark.table("assignment_techmart.analytics.customer_summary").count()
        if row_count > 0:
            print(f"[PASS] Task 2: Summary table has {row_count} rows of data")
            checks_passed += 1
        else:
            print("[FAIL] Task 2: Summary table is empty")
    else:
        print("[SKIP] Task 2: Table doesn't exist")
except Exception as e:
    print(f"[FAIL] Task 2: Error counting rows - {e}")

# Check 4: Orders table has data
checks_total += 1
try:
    order_count = spark.table("assignment_techmart.analytics.orders").count()
    if order_count > 0:
        print(f"[PASS] Setup: Orders table has {order_count} rows")
        checks_passed += 1
    else:
        print("[FAIL] Setup: Orders table is empty")
except Exception as e:
    print(f"[FAIL] Setup: Error - {e}")

# Check 5: All source tables exist
checks_total += 1
try:
    all_exist = all([
        spark.catalog.tableExists("assignment_techmart.analytics.customers"),
        spark.catalog.tableExists("assignment_techmart.analytics.products"),
        spark.catalog.tableExists("assignment_techmart.analytics.orders")
    ])
    if all_exist:
        print("[PASS] Setup: All source tables exist (customers, products, orders)")
        checks_passed += 1
    else:
        print("[FAIL] Setup: Some source tables are missing")
except Exception as e:
    print(f"[FAIL] Setup: Error - {e}")

print("=" * 60)
print(f"RESULT: {checks_passed}/{checks_total} automated checks passed")
print("=" * 60)
print("""
MANUAL CHECKS (verify by visual inspection):
[ ] Task 1: Query uses at least one JOIN
[ ] Task 1: Query uses at least one window function (RANK, LAG, SUM OVER, etc.)
[ ] Task 1: Query filters by a date range
[ ] Task 3: Visualization displays correctly
[ ] Task 4: Alert query returns a single numeric value
""")

---
# Assignment 2: Production RAG Agent

## Scenario
You are a Generative AI Engineer at **SupportAI**, a company building a customer support agent for e-commerce clients. The agent must answer customer questions across three categories:

1. **Product information** ("What features does Pro License include?") → RAG retrieval from product docs
2. **Order status** ("Where is my order #12345?") → Tool/function call to live order database
3. **General questions** ("What are your business hours?") → Direct LLM response with system prompt

## Constraints
Your agent MUST implement ALL of the following:

| # | Constraint | Demo Reference |
|---|---|---|
| 1 | **Vector Search retrieval** for product docs with at least 5 chunks indexed | Demo 3 |
| 2 | **Tool/function calling** with JSON schema validation for order lookup | Demo 5 |
| 3 | **Retrieval gating** — refuse to answer if best chunk similarity < 0.5 | Demo 3 |
| 4 | **PII guardrails** — use `ai_mask` to redact PII before sending to LLM | Demo 4 |
| 5 | **Multi-step routing** — classify question type, then route to appropriate handler | Demo 5 |
| 6 | **LLM-as-judge evaluation** — evaluate 3 sample Q&A pairs on faithfulness & relevance | Demo 7 |
| 7 | **Unity Catalog governance** — all data in UC tables, not local files | Demo 0/7 |
| 8 | **System prompt with refusal policy** — "I don't know" when evidence is weak | Demo 3 |

## Deliverables
Complete each task cell below. The validation cell at the bottom checks that all constraints are met.

> **Tip**: Review Demo 3 (Vector Search), Demo 4 (AI Gateway/PII), Demo 5 (Agents/Tools), and Demo 7 (Evaluation) for patterns you can adapt.

In [0]:
# SETUP: Run this cell to create sample data for Assignment 2
# Creates: product docs for RAG, order database for tool calling, evaluation dataset

from pyspark.sql.functions import explode, array, lit, explode, sequence, rand, col

# 1. Create UC catalog/schema for the agent
spark.sql("CREATE CATALOG IF NOT EXISTS assignment_supportai")
spark.sql("CREATE SCHEMA IF NOT EXISTS assignment_supportai.agent")

# 2. Product documentation chunks (for Vector Search indexing)
product_docs = [
    (1, "Pro License", "Pro License includes up to 50 concurrent users, priority email support, and access to the dashboard builder. Maximum data limit is 10GB per workspace.", "product_info"),
    (2, "Pro License", "Pro License pricing starts at $499/month for annual billing or $599/month for monthly billing. Volume discounts available for 10+ licenses.", "pricing"),
    (3, "Enterprise License", "Enterprise License includes unlimited users, 24/7 phone support, dedicated CSM, custom integrations, and SSO. No data limits.", "product_info"),
    (4, "Enterprise License", "Enterprise License pricing starts at $1,499/month. Custom pricing available for 100+ seats. Contact sales for a quote.", "pricing"),
    (5, "Vector Search", "Vector Search enables semantic similarity search over your documents. Supports Delta Sync indexes with auto-embedding. Use AI Search for managed RAG.", "product_info"),
    (6, "Vector Search", "Vector Search pricing is based on DBU consumption of the vector search endpoint. Storage-optimized endpoints available for 100M+ vectors.", "pricing"),
    (7, "AI Gateway", "AI Gateway provides traffic routing, rate limiting, PII guardrails, and usage tracking for LLM endpoints. Supports fallback and canary deployments.", "product_info"),
    (8, "AI Gateway", "AI Gateway guardrails include ai_mask for PII redaction, content filters, and profanity filters. All guardrails are configurable per-endpoint.", "security"),
    (9, "Support Plan", "Support Plan includes business hours support (9 AM - 6 PM EST), 48-hour response time SLA, and access to the knowledge base.", "product_info"),
    (10, "Support Plan", "Support Plan costs $99/month per workspace. Enterprise support (24/7) available with Enterprise License only.", "pricing"),
]

spark.createDataFrame(product_docs, ["chunk_id", "product", "chunk_text", "chunk_type"]) \
    .write.mode("overwrite").saveAsTable("assignment_supportai.agent.product_docs")

# 3. Orders table (for tool/function calling)
orders_data = [
    (12345, "Alice Corp",     "Shipped",  "2025-09-20", "FedEx",  "TRK123456", "2025-09-25"),
    (12346, "Bob Industries", "Processing", "2025-09-22", None,     None,        None),
    (12347, "Charlie LLC",    "Delivered", "2025-09-15", "UPS",    "TRK789012", "2025-09-21"),
    (12348, "Delta Inc",      "Cancelled", "2025-09-18", None,     None,        None),
    (12349, "Echo Partners",   "Shipped",  "2025-09-23", "DHL",    "TRK456789", "2025-09-27"),
]

spark.createDataFrame(orders_data, ["order_id", "customer_name", "status", "order_date", "carrier", "tracking_number", "estimated_delivery"]) \
    .write.mode("overwrite").saveAsTable("assignment_supportai.agent.orders")

# 4. Evaluation dataset (for LLM-as-judge in Task 6)
eval_data = [
    (1, "What features does Pro License include?", "Pro License includes up to 50 concurrent users, priority email support, dashboard builder, and 10GB data limit per workspace."),
    (2, "Where is my order 12345?", "Your order 12345 has been shipped via FedEx with tracking number TRK123456. Estimated delivery is September 25, 2025."),
    (3, "What are your business hours?", "Our support team is available Monday through Friday, 9 AM to 6 PM EST. Enterprise customers have 24/7 support."),
]

spark.createDataFrame(eval_data, ["eval_id", "question", "expected_answer"]) \
    .write.mode("overwrite").saveAsTable("assignment_supportai.agent.eval_dataset")

print("✅ Setup complete! Tables created:")
print("  - assignment_supportai.agent.product_docs (10 chunks)")
print("  - assignment_supportai.agent.orders (5 orders)")
print("  - assignment_supportai.agent.eval_dataset (3 Q&A pairs)")

## Task 1: Vector Search Index for Product Docs

**Constraint**: Index at least 5 product doc chunks for semantic retrieval.

**Instructions**:
1. Create a Vector Search endpoint (or use an existing one)
2. Create a Delta Sync index on `assignment_supportai.agent.product_docs`
3. The source table needs a primary key and Change Data Feed enabled
4. Use `databricks-gte-large-en` as the embedding model
5. Sync the index and verify it returns results

> **Hint**: See Demo 3 for the full Vector Search creation pattern. You'll need `VectorSearchClient` from the Databricks SDK.
>
> **Free Edition note**: If you can't create a new Vector Search endpoint, write the code that WOULD create it and add a comment explaining what you would do. The validation checks for the code structure, not the actual index.

In [0]:
# TASK 1: Create Vector Search Index for Product Documentation
#
# CONSTRAINT: Must index at least 5 chunks for semantic retrieval.
#
# -- PASTE YOUR CODE BELOW

## Task 2: Tool/Function Calling for Order Lookup

**Constraint**: Define a tool with JSON schema validation that looks up order status by order_id.

**Instructions**:
1. Define a Python function `lookup_order(order_id: int) -> dict` that queries `assignment_supportai.agent.orders`
2. Define a JSON schema for the tool (parameter: `order_id` as integer, required)
3. The function should return order status, carrier, tracking number, and estimated delivery
4. Include input validation (order_id must be a positive integer)
5. Handle the case where order_id doesn't exist (return "Order not found")

> **Hint**: See Demo 5 for tool definition patterns. The JSON schema should constrain the model to valid order IDs only.

In [0]:
# TASK 2: Tool/Function Calling for Order Lookup
#
# CONSTRAINT: Must define a tool with JSON schema validation.
#
# -- PASTE YOUR CODE BELOW

## Task 3: Retrieval Gating

**Constraint**: Implement retrieval gating with a minimum similarity threshold of 0.5. If the best chunk's similarity score is below 0.5, the agent must refuse to answer ("I don't know. I couldn't find relevant information.").

**Instructions**:
1. After Vector Search retrieval, check the similarity score of the top result
2. If score < 0.5, return a refusal message instead of passing context to the LLM
3. If score >= 0.5, proceed with context-augmented generation
4. Log the similarity score for debugging

> **Hint**: See Demo 3 notes on retrieval gating. The threshold check should happen BEFORE the LLM call, not after.

In [0]:
# TASK 3: Retrieval Gating with Minimum Similarity Threshold
#
# -- PASTE YOUR CODE BELOW

## Task 4: PII Guardrails with ai_mask

**Constraint**: Use `ai_mask` to redact PII from user inputs before sending to the LLM.

**Instructions**:
1. Write a function that uses the `ai_mask` SQL function to redact PII (names, emails, phone numbers) from user queries
2. Apply masking BEFORE the query is sent to the LLM for generation
3. The masked query should still preserve enough context for the agent to understand the question
4. Log the original and masked query for audit purposes

> **Hint**: See Demo 4 for the ai_mask pattern. Use `spark.sql("SELECT ai_mask(...)").` The ai_mask function takes content and a list of PII types to mask.

In [0]:
# TASK 4: PII Guardrails with ai_mask
#
# -- PASTE YOUR CODE BELOW

## Task 5: Multi-Step Routing Agent

**Constraint**: Implement question-type classification and routing to 3 handlers. Include a system prompt with refusal policy.

**Instructions**:
1. Define a system prompt that:
   - Instructs the agent to classify the question type (product_info, order_status, general)
   - Includes a refusal policy: "If you cannot find relevant information, say 'I don't know.'"
   - Defines what each question type means
2. Define a routing function `route_question(question: str) -> str` that:
   - Uses an LLM to classify the question type
   - Routes product_info → RAG retrieval (Task 1) with gating (Task 3)
   - Routes order_status → Tool call (Task 2)
   - Routes general → Direct LLM response
   - Applies PII masking (Task 4) to all inputs
3. Define a main `agent_answer(question: str) -> str` function that:
   - Masks PII
   - Routes the question
   - Calls the appropriate handler
   - Returns the answer

> **Hint**: See Demo 5 for multi-step LLM workflow patterns. The classification step can use `ai_query` with a prompt like "Classify this question as product_info, order_status, or general: {question}".

In [0]:
# TASK 5: Multi-Step Routing Agent with System Prompt
#
# -- PASTE YOUR CODE BELOW

## Task 6: LLM-as-Judge Evaluation

**Constraint**: Evaluate 3 Q&A pairs from the eval dataset using LLM-as-judge on faithfulness and relevance.

**Instructions**:
1. Load the evaluation dataset from `assignment_supportai.agent.eval_dataset`
2. For each question, get the agent's answer using `agent_answer()` from Task 5
3. Use `ai_query` with an LLM to evaluate each answer on two dimensions:
   - **Faithfulness**: Is the answer grounded in the retrieved context / tool data? (1-5 scale)
   - **Relevance**: Is the answer relevant to the question asked? (1-5 scale)
4. Store evaluation results in a UC table: `assignment_supportai.agent.eval_results`
5. Print a summary of the evaluation scores

> **Hint**: See Demo 7 for LLM-as-judge patterns. The judge prompt should ask the LLM to score the answer on faithfulness and relevance, then return a structured score.

In [0]:
# TASK 6: LLM-as-Judge Evaluation
#
# -- PASTE YOUR CODE BELOW

In [0]:
# VALIDATION: Run this cell AFTER completing all 6 tasks to check constraints.
# This cell verifies that your solution meets all the requirements.

print("=" * 60)
print("ASSIGNMENT 2 - VALIDATION CHECKS")
print("=" * 60)

checks_passed = 0
checks_total = 0

# Check 1: Product docs table exists with >= 5 chunks
checks_total += 1
try:
    if spark.catalog.tableExists("assignment_supportai.agent.product_docs"):
        count = spark.table("assignment_supportai.agent.product_docs").count()
        if count >= 5:
            print(f"[PASS] Task 1: Product docs table has {count} chunks (>= 5 required)")
            checks_passed += 1
        else:
            print(f"[FAIL] Task 1: Product docs table has only {count} chunks (need >= 5)")
    else:
        print("[FAIL] Task 1: Product docs table doesn't exist")
except Exception as e:
    print(f"[FAIL] Task 1: Error - {e}")

# Check 2: Orders table exists
checks_total += 1
try:
    if spark.catalog.tableExists("assignment_supportai.agent.orders"):
        print("[PASS] Task 2: Orders table exists for tool calling")
        checks_passed += 1
    else:
        print("[FAIL] Task 2: Orders table doesn't exist")
except Exception as e:
    print(f"[FAIL] Task 2: Error - {e}")

# Check 3: Eval dataset exists with >= 3 pairs
checks_total += 1
try:
    if spark.catalog.tableExists("assignment_supportai.agent.eval_dataset"):
        count = spark.table("assignment_supportai.agent.eval_dataset").count()
        if count >= 3:
            print(f"[PASS] Task 6: Eval dataset has {count} Q&A pairs (>= 3 required)")
            checks_passed += 1
        else:
            print(f"[FAIL] Task 6: Eval dataset has only {count} pairs (need >= 3)")
    else:
        print("[FAIL] Task 6: Eval dataset doesn't exist")
except Exception as e:
    print(f"[FAIL] Task 6: Error - {e}")

# Check 4: Eval results table exists (student must create this in Task 6)
checks_total += 1
try:
    if spark.catalog.tableExists("assignment_supportai.agent.eval_results"):
        print("[PASS] Task 6: Eval results table exists in UC")
        checks_passed += 1
    else:
        print("[FAIL] Task 6: Eval results table doesn't exist (create it in Task 6)")
except Exception as e:
    print(f"[FAIL] Task 6: Error - {e}")

# Check 5: All data is in Unity Catalog (not local files)
checks_total += 1
try:
    tables = [t for t in spark.catalog.listTables("assignment_supportai.agent")]
    uc_tables = [t.name for t in tables if t.tableType != 'TEMPORARY']
    if len(uc_tables) >= 3:
        print(f"[PASS] Task 7: {len(uc_tables)} UC tables found: {uc_tables}")
        checks_passed += 1
    else:
        print(f"[FAIL] Task 7: Only {len(uc_tables)} UC tables found (need >= 3)")
except Exception as e:
    print(f"[FAIL] Task 7: Error - {e}")

print("=" * 60)
print(f"RESULT: {checks_passed}/{checks_total} automated checks passed")
print("=" * 60)
print("""
MANUAL CHECKS (verify by visual inspection):
[ ] Task 1: Vector Search index created with databricks-gte-large-en
[ ] Task 2: lookup_order function defined with JSON schema validation
[ ] Task 3: Retrieval gating implemented with 0.5 threshold
[ ] Task 4: ai_mask used for PII redaction before LLM call
[ ] Task 5: route_question classifies into 3 types (product_info, order_status, general)
[ ] Task 5: System prompt includes refusal policy ("I don't know")
[ ] Task 5: agent_answer function integrates all components (mask -> route -> handle -> respond)
[ ] Task 6: LLM-as-judge evaluates faithfulness AND relevance (both scored)
[ ] Task 6: Results stored in UC table, not local variable only
""")

---
# Grading Rubric

## Assignment 1: SQL Analytics Dashboard (40 points)

| Criterion | Points | How to Score |
|---|---|---|
| JOIN across 2+ tables | 8 | Query must join customers, products, and/or orders |
| Window function used | 8 | RANK, LAG, SUM OVER, or similar |
| Date range filter | 5 | WHERE clause filtering by date |
| Derived metric calculated | 8 | Profit margin %, revenue growth %, or AOV |
| UC table (not temp view) | 5 | CREATE TABLE or saveAsTable in UC catalog |
| Alert query returns single value | 3 | Query returns one numeric column for alert |
| Visualization created | 3 | Easily understandable Dashboard based on Screenshots |

## Assignment 2: Production RAG Agent (60 points)

| Criterion | Points | How to Score |
|---|---|---|
| Vector Search index with 5+ chunks | 10 | Index created on product_docs, sync successful |
| Tool with JSON schema validation | 8 | lookup_order function + schema dict defined |
| Retrieval gating (threshold 0.5) | 8 | Score check before LLM call, refusal on low score |
| PII guardrails (ai_mask) | 8 | ai_mask called before LLM generation |
| Multi-step routing (3 types) | 12 | route_question classifies + agent_answer routes correctly |
| System prompt with refusal policy | 6 | "I don't know" policy in system prompt |
| LLM-as-judge (faithfulness + relevance) | 6 | Both dimensions scored for 3 Q&A pairs |
| Eval results in UC table | 2 | assignment_supportai.agent.eval_results exists |

**Total: 100 points**

---

## Clean Up

When you're done, run this to clean up all assignment artifacts:

```sql
DROP CATALOG IF EXISTS assignment_techmart CASCADE;
DROP CATALOG IF EXISTS assignment_supportai CASCADE;
```